# Optional Project - Transformer Scaling Laws on SVGs

Vincent Hepola - vh2308


# 0. Imports


In [1]:
from IPython.display import SVG, display
import matplotlib.pyplot as plt

import os

# Replace '/path/to/new/cache' with your desired directory
os.environ["HF_HOME"] = "./.cache"
print(os.environ["HF_HOME"])

from datasets import load_dataset, concatenate_datasets, DatasetDict, Value

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


from torch.optim import AdamW
from pl_bolts.optimizers.lr_scheduler import LinearWarmupCosineAnnealingLR

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

from lxml import etree
import cairosvg

import numpy as np
from tqdm.auto import tqdm
from scipy.optimize import curve_fit


import re
import time

import mup
from mup import MuReadout, make_base_shapes, set_base_shapes, MuSGD, MuAdam

./.cache


c:\Users\Vince\miniconda3\envs\neuroinfo\Lib\site-packages\pl_bolts\__init__.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
c:\Users\Vince\miniconda3\envs\neuroinfo\Lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
c:\Users\Vince\miniconda3\envs\neuroinfo\Lib\site-packages\pl_bolts\models\self_supervised\amdim\amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lig

# 1. Data Collecting and Preprocessing


## 1.1 Datasets


In [2]:
# dataset_1 = load_dataset("starvector/svg-icons-simple")
# dataset_1 = dataset_1.cast_column("Filename", Value("string"))

# dataset_2 = load_dataset("starvector/svg-emoji-simple")
# dataset_2 = dataset_2.cast_column("Filename", Value("string"))

dataset_3 = load_dataset("starvector/svg-stack-simple")
dataset_3 = dataset_3.cast_column("Filename", Value("string"))

# Combine all splits using a dictionary comprehension
combined_dataset = DatasetDict(
    {
        split: concatenate_datasets(
            # [dataset_1[split], dataset_2[split], dataset_3[split]]
            [dataset_3[split]]
        )
        # for split in dataset_1.keys()
        for split in dataset_3.keys()
    }
)

total_items = 300000

dataset = combined_dataset["train"].select(range(total_items))

print(dataset)

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['Filename', 'Svg'],
    num_rows: 300000
})


### View Sample


In [3]:
# sample_1 = dataset["Svg"][2]

In [4]:
# print(sample_1)
# display(SVG(sample_1))

## 1.2 Normalizing/Cleaning

- Strip comments, metadata, and unnecessary whitespace
- Normalize coordinate precision (e.g., round to 1 decimal place) to reduce vocabulary
- Optionally canonicalize attribute ordering
- Remove SVGs that are too short (< 50 characters) or too long (above your chosen token threshold)
- Validate that all SVGs in the cleaned set parse as valid XML
- Ensure all SVGs render without errors (use lxml for XML parsing, optionally CairoSVG for render validation)


In [5]:
def clean_svg(svg_text):
    # 1. Remove XML comments
    svg_text = re.sub(r"", "", svg_text, flags=re.DOTALL)

    # 2. Function to clean ONLY the path data string
    def clean_path_data(match):
        path_string = match.group(1)

        # Space out the letters (M, L, C, Z, etc.)
        path_string = re.sub(r"([a-zA-Z])", r" \1 ", path_string)

        # Replace commas with spaces
        path_string = path_string.replace(",", " ")

        # Round numbers to 1 decimal place
        def round_match(m):
            try:
                return f"{float(m.group(0)):.1f}"
            except ValueError:
                return m.group(0)

        path_string = re.sub(r"-?\d*\.?\d+", round_match, path_string)

        # Clean up any messy double-spaces inside the quotes
        path_string = re.sub(r"\s+", " ", path_string).strip()

        # Return it wrapped back in d="..."
        return f'd="{path_string}"'

    # 3. Apply the cleaner ONLY to the d="..." attributes
    svg_text = re.sub(r'd="([^"]+)"', clean_path_data, svg_text)

    # 4. Clean up any overall messy whitespace
    svg_text = re.sub(r"\s+", " ", svg_text).strip()

    return svg_text


# # Example Test:
# raw_svg = '<svg><path d="M10,20.35L.5 30"/></svg>'
# print(clean_svg(raw_svg))

In [6]:
# svg_string = clean_svg(sample_1)
# print(svg_string)

### Clean and verify all svgs in training set


In [7]:
TOKEN_THRESHOLD = 2048

In [8]:
from svg_utils import process_row

In [9]:
print("Cleaning datasets with multiprocessing...")
cleaned_dataset = dataset.map(process_row, num_proc=8)
print("Done!")

remaining_items = len(cleaned_dataset)
print(remaining_items)

Cleaning datasets with multiprocessing...
Done!
300000


In [10]:
stop_1 = int(remaining_items * 0.98)
stop_2 = stop_1 + int(remaining_items * 0.01)
stop_3 = stop_2 + int(remaining_items * 0.01)

cleaned_train = cleaned_dataset.select(range(stop_1))
cleaned_val = cleaned_dataset.select(range(stop_1, stop_2))
cleaned_test = cleaned_dataset.select(range(stop_2, stop_3))

In [11]:
cleaned_train

Dataset({
    features: ['Filename', 'Svg'],
    num_rows: 294000
})

In [12]:
cleaned_val

Dataset({
    features: ['Filename', 'Svg'],
    num_rows: 3000
})

In [13]:
cleaned_test

Dataset({
    features: ['Filename', 'Svg'],
    num_rows: 3000
})

In [14]:
def is_valid(d):
    svg_string = d["Svg"]

    if len(svg_string) <= 50 or len(svg_string) > TOKEN_THRESHOLD:
        return False

    try:
        etree.fromstring(svg_string.encode("utf-8"))
    except etree.XMLSyntaxError:
        return False

    return True


# valid_render runs very long and hasn't filtered out any svg yet... SKIP!

# def valid_render(d):
#     svg_string = d["Svg"]

#     try:
#         cairosvg.svg2png(bytestring=svg_string.encode("utf-8"))
#     except:
#         return False

#     return True


filtered_train = cleaned_train.filter(is_valid)
# filtered_train = filtered_train.filter(valid_render)

In [15]:
print(f"filtered_train.num_rows: {filtered_train.num_rows}")

filtered_train.num_rows: 204630


## 1.3 Tokenize

Train a BPE tokenizer on your SVG corpus using sentencepiece or the HuggingFace
tokenizers library. Vocabulary sizes in the range 1K–8K are reasonable. Document the vocabulary
size and justify your choice in the report.


In [16]:
# small initials

VOCAB_SIZE = 1000
BLOCK_SIZE = 256

In [17]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

In [18]:
try:
    tokenizer = Tokenizer.from_file("tokenizer.json")
except:
    print("No tokenizer found in root!")

    tokenizer = Tokenizer(models.BPE())

    tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
        [pre_tokenizers.WhitespaceSplit(), pre_tokenizers.Punctuation()]
    )

    trainer = trainers.BpeTrainer(
        special_tokens=["<|endoftext|>", "<|pad|>"], vocab_size=VOCAB_SIZE
    )

    tokenizer.train_from_iterator(filtered_train["Svg"], trainer)

    tokenizer.save("tokenizer.json")

In [19]:
# sample_2 = filtered_train["Svg"][0]
# sample_2

In [20]:
# encoded_sample_2 = tokenizer.encode(sample_2)

# print(encoded_sample_2)


# # First 10 tokens on encoding
# for id, token in zip(encoded_sample_2.ids[:10], encoded_sample_2.tokens[:10]):
#     print(f"{id} -> {token}")

In [21]:
# tokenizer.decode(encoded_sample_2.ids)

In [22]:
def tokenize_svg(d):

    svg_string = d["Svg"]

    ids = tokenizer.encode(svg_string).ids

    eot_token = tokenizer.token_to_id("<|endoftext|>")

    ids.append(eot_token)

    return {"Filename": d["Filename"], "Svg": d["Svg"], "input_ids": ids}

In [23]:
tokenized_train = filtered_train.map(tokenize_svg)

In [24]:
# tokenized_train["input_ids"]

In [25]:
def flatten_input_ids(tokenized_dataset):
    concat_arr = []

    for arr in tqdm(tokenized_dataset["input_ids"]):
        concat_arr.extend(arr)

    return concat_arr


train_input_ids = flatten_input_ids(tokenized_train)
train_input_ids = np.array(train_input_ids)

len(train_input_ids)

  0%|          | 0/204630 [00:00<?, ?it/s]

130428077

In [26]:
print(f"len(np.unique(train_input_ids)): {len(np.unique(train_input_ids))}")

len(np.unique(train_input_ids)): 940


### Create, clean, filter, and tokenize test/val datasets


In [27]:
filtered_test = cleaned_test.filter(is_valid)
# filtered_val = filtered_val.filter(valid_render)
tokenized_test = filtered_test.map(tokenize_svg)
test_input_ids = flatten_input_ids(tokenized_test)
test_input_ids = np.array(test_input_ids)

filtered_val = cleaned_val.filter(is_valid)
# filtered_val = filtered_val.filter(valid_render)
tokenized_val = filtered_val.map(tokenize_svg)
val_input_ids = flatten_input_ids(tokenized_val)
val_input_ids = np.array(val_input_ids)

  0%|          | 0/2066 [00:00<?, ?it/s]

  0%|          | 0/2088 [00:00<?, ?it/s]

In [28]:
# Check the size of remaining data
all_tokens = np.concat([train_input_ids, val_input_ids, test_input_ids])
print(f"Number of tokens in data: {len(all_tokens)}")
print(
    f"Number of tokens in train: {len(train_input_ids)} -> {(len(train_input_ids)/len(all_tokens)*100):.2f}%"
)
print(
    f"Number of tokens in val: {len(val_input_ids)} -> {(len(val_input_ids)/len(all_tokens)*100):.2f}%"
)
print(
    f"Number of tokens in test: {len(test_input_ids)} -> {(len(test_input_ids)/len(all_tokens)*100):.2f}%"
)

Number of tokens in data: 133084710
Number of tokens in train: 130428077 -> 98.00%
Number of tokens in val: 1355632 -> 1.02%
Number of tokens in test: 1301001 -> 0.98%


# 2. Transformer Scaling Study

Train a family of decoder-only transformer language models of varying sizes on the SVG data. Measure the
validation loss after 1 epoch of training for each model size.


## 2.0 Setup


### 2.0.0 Dataloaders


In [39]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Convert to 1D tensors
train_data = torch.tensor(train_input_ids, dtype=torch.long)
val_data = torch.tensor(val_input_ids, dtype=torch.long)

# We want sequences of length (BLOCK_SIZE + 1)
# Assuming you still have BLOCK_SIZE = 256 from your earlier cell
seq_len = BLOCK_SIZE + 1


def chunk_data(data, seq_len):
    # Calculate how many full sequences we can make
    num_chunks = len(data) // seq_len

    # Truncate any leftover tokens at the very end that don't fit into a full chunk
    data = data[: num_chunks * seq_len]

    # Reshape the 1D tensor into a 2D grid of shape (num_chunks, seq_len)
    return data.view(num_chunks, seq_len)


# 2. Reshape 1D streams into 2D grids
train_data_2d = chunk_data(train_data, seq_len)
val_data_2d = chunk_data(val_data, seq_len)

# 3. NOW we can slice into X and Y safely!
# X gets everything except the last token of each chunk
X_train = train_data_2d[:, :-1]
# Y gets everything except the first token of each chunk
Y_train = train_data_2d[:, 1:]

X_val = val_data_2d[:, :-1]
Y_val = val_data_2d[:, 1:]

# 4. Bundle into TensorDataset and DataLoader
train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

print(f"X Train shape: {X_train.shape}, Y Train shape: {Y_train.shape}")
print(f"X Val shape: {X_val.shape}, Y Val shape: {Y_val.shape}")

X Train shape: torch.Size([507502, 256]), Y Train shape: torch.Size([507502, 256])
X Val shape: torch.Size([5274, 256]), Y Val shape: torch.Size([5274, 256])


### 2.0.1 Define Model and Functions


In [ ]:
def estimate_loss(model, val_loader, eval_iters=50):

    model.eval()
    with torch.inference_mode():
        losses = torch.zeros(eval_iters)
        val_iter = iter(val_loader)

        for k in range(eval_iters):
            try:
                X, Y = next(val_iter)
            except StopIteration:
                val_iter = iter(val_loader)
                X, Y = next(val_iter)

            X, Y = X.to(device), Y.to(device)

            _, loss = model(X, targets=Y)

            losses[k] = loss.item()

    model.train()
    return losses.mean().item()


def train_loop(
    model, optimizer, train_loader, val_loader, steps, scheduler=None, eval_interval=100
):
    model.train()
    train_iter = iter(train_loader)

    # --- METRICS TRACKING ---
    train_loss_history = []
    val_loss_history = []  # <--- NEW: List to hold validation losses
    eval_steps = []  # <--- NEW: List to hold the step numbers for validation
    total_tokens_processed = 0
    start_time = time.time()

    # Reset GPU memory stats if using CUDA
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    for step in range(steps):
        try:
            X, Y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            X, Y = next(train_iter)

        X, Y = X.to(device), Y.to(device)
        total_tokens_processed += X.numel()  # Batch_size * Sequence_length

        _, loss = model(X, targets=Y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )  # Clipping gradients
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        train_loss_history.append(loss.item())

        if step % eval_interval == 0 or step == steps - 1:
            current_lr = optimizer.param_groups[0]["lr"]
            val_loss = estimate_loss(model, val_loader)

            val_loss_history.append(val_loss)
            eval_steps.append(step)

            print(
                f"Step {step:04d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f} | lr: {current_lr:.9e}"
            )

    # --- FINALIZE METRICS ---
    end_time = time.time()
    wall_clock_time = end_time - start_time
    tokens_per_sec = total_tokens_processed / wall_clock_time

    max_mem_mb = 0
    if torch.cuda.is_available():
        max_mem_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    final_val_loss = estimate_loss(model, val_loader)

    return {
        "final_val_loss": final_val_loss,
        "train_loss_history": train_loss_history,
        "val_loss_history": val_loss_history,  # <--- NEW: Return the list
        "eval_steps": eval_steps,  # <--- NEW: Return the step numbers
        "wall_clock_time": wall_clock_time,
        "tokens_per_sec": tokens_per_sec,
        "gpu_memory_mb": max_mem_mb,
    }

In [32]:
# Keeping depth the same (n_layers, n_heads) and only varying width (d_model, d_ff)
configs = {
    "Tiny": {"d_model": 128, "n_layers": 6, "n_heads": 6, "d_ff": 512},
    "XL": {"d_model": 768, "n_layers": 6, "n_heads": 6, "d_ff": 3072},
}


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 4. Best Model Training and Sample Generation


In Part 2, you tuned the learning rate on the smallest model and used it for all model sizes. In this part,
you will investigate whether µP (Maximal Update Parameterization) can improve scaling behavior by
enabling principled learning rate transfer across model widths. You will also use your scaling laws to make
predictions beyond the model sizes you trained.


## 3.1 Scaling Study


### 3.1.1 Reparameterize


In [ ]:
class MuHead(nn.Module):
    """one head of self-attention"""

    def __init__(self, head_size, d_model, block_size):
        super().__init__()
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

        self.head_size = head_size  # Store this

        # 'tril' is the lower triangular matrix for masking
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # Batch, sequence length, embedding dimensionality (d_model)

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Compute attention scores
        wei = q @ k.transpose(-2, -1) * (self.head_size**-1)

        # Mask out future tokens by replacing 0s in the tril matrix with -infinity
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        output = F.softmax(wei, dim=-1) @ v

        return output


class MuMultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, block_size):
        super().__init__()
        head_size = d_model // n_heads
        self.heads = nn.ModuleList(
            [MuHead(head_size, d_model, block_size) for _ in range(n_heads)]
        )

        self.proj = nn.Linear(head_size * n_heads, d_model)

    def forward(self, x):
        x = torch.concat([head(x) for head in self.heads], dim=-1)
        x = self.proj(x)
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()

        if d_ff == None:
            d_ff = 4 * d_model

        self.layer = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.layer(x)


class MuTransformerBlock(nn.Module):
    def __init__(self, n_heads, d_model, d_ff, block_size):
        super().__init__()

        self.MHA = MuMultiHeadAttention(
            n_heads=n_heads, d_model=d_model, block_size=block_size
        )
        self.ff = FeedForward(d_model=d_model, d_ff=d_ff)
        self.ln_1 = nn.LayerNorm(d_model)
        self.ln_2 = nn.LayerNorm(d_model)

    def forward(self, x):

        x = x + self.MHA(self.ln_1(x))
        x = x + self.ff(self.ln_2(x))

        return x


class MuCustomTransformer(nn.Module):
    """
    n_layers: The total number of identical Transformer blocks stacked on top of each other.
    n_heads: The number of parallel "attention heads" used in the Multi-Head Attention mechanism.
    d_model: The dimensionality of the input and output embeddings, also known as the "hidden size" or "embedding size".
    d_ff: The size of the hidden dimension in the position-wise feed-forward networks (MLP) within each layer.
    """

    def __init__(self, vocab_size, block_size, n_layers, n_heads, d_model, d_ff):
        super().__init__()

        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_ff = d_ff

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = nn.Embedding(block_size, d_model)

        self.blocks = nn.Sequential(
            *[
                MuTransformerBlock(n_heads, d_model, d_ff, block_size)
                for _ in range(n_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.proj = mup.MuReadout(d_model, vocab_size)

    def generate(self, idx, max_new_tokens, temperature=0.8):
        # idx.shape (B,T)

        for _ in range(max_new_tokens):

            # Crop to block_size if too long
            if idx.size(1) <= self.block_size:
                idx_continue = idx
            else:
                idx_continue = idx[:, -self.block_size :]

            # forward into model.
            logits, _ = self(idx_continue)

            # Get the final step logits
            logits = logits[:, -1, :] / temperature

            probs = F.softmax(logits, dim=-1)

            # sample randomly from the distribution
            # print(probs.shape)
            idx_next = torch.multinomial(probs, num_samples=1)

            # append the new index to the sequence and continue
            idx = torch.concat((idx, idx_next), dim=1)

        return idx

    def forward(self, idx, targets=None):

        sequence_length = idx.shape[1]

        te = self.token_embedding(idx)

        pe = self.positional_embedding(torch.arange(sequence_length, device=idx.device))

        x = te + pe  # input_embedding

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.proj(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [34]:
mu_best_lr = 1e-2

print(f"Lowest val_loss when lr = {mu_best_lr}")

Lowest val_loss when lr = 0.01


In [ ]:
mup_training_results = {}

steps_per_epoch = len(train_loader) // 92
epochs = 2

total_training_steps = steps_per_epoch * epochs


name = "XL"  # Explicitly defining the name for your print statements and saving

print(
    f" Starting muP Training {name} model with Best LR = {mu_best_lr}\n"
    f" (1 Epoch = {steps_per_epoch} steps | Total = {total_training_steps} steps)"
)

# Initialize the proxy (Tiny) config
base_model = MuCustomTransformer(
    vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, **configs["Tiny"]
).to(device)

# Initialize the model with the specific config
target_model = MuCustomTransformer(
    vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, **configs[name]
).to(device)

mup.set_base_shapes(target_model, base_model)

target_model.apply(
    lambda m: (
        mup.init.kaiming_uniform_(m.weight, a=5**0.5)
        if hasattr(m, "weight") and m.weight.dim() > 1
        else None
    )
)

params = count_parameters(target_model)
print(f"Parameters: {params:,}")


print(f"\n{'='*40}")
print(f"Training {name} Model for {epochs} Epochs")
print(f"{'='*40}")

optimizer = mup.MuAdamW(target_model.parameters(), lr=mu_best_lr)

scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_epochs=int(total_training_steps * 0.05),  # Warm up for first 5% of all steps
    max_epochs=total_training_steps,  # Stretch decay over all 50 epochs
    warmup_start_lr=0.0,
    eta_min=mu_best_lr * 0.1,  # Decay to 10% of max LR
)

for epoch in range(epochs):
    print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

    metrics = train_loop(
        target_model,
        optimizer,
        train_loader,
        val_loader,
        steps=steps_per_epoch,
        scheduler=scheduler,
        eval_interval=200,
    )

    # Print the requested tracking metrics
    print(f"   Epoch Val Loss: {metrics['final_val_loss']:.4f}")
    print(f"   Wall-clock time: {metrics['wall_clock_time']:.1f} sec")
    print(f"   Throughput: {metrics['tokens_per_sec']:,.0f} tokens/sec")
    print(f"   Peak GPU Memory: {metrics['gpu_memory_mb']:.1f} MB")

    # Save for plotting
    metrics["params"] = params
    mup_training_results[name] = metrics

    # Optional: Save the model weights so you can generate SVGs later
    torch.save(target_model.state_dict(), f"mu_model_XL_best.pt")

 Starting muP Training XL model with Best LR = 0.01
 (1 Epoch = 86 steps | Total = 172 steps)
Parameters: 44,248,552

Training XL Model for 2 Epochs

--- Epoch 1/2 ---


C:\Users\Vince\AppData\Local\Temp\ipykernel_22700\278199327.py:46: UnderReviewWarning: The feature LinearWarmupCosineAnnealingLR is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  scheduler = LinearWarmupCosineAnnealingLR(


Step 0000 | Train Loss: 6.9759 | Val Loss: 6.9733 | lr: 2.380952381e-04
Step 0085 | Train Loss: 2.0505 | Val Loss: 2.0433 | lr: 1.358849751e-03
   Epoch Val Loss: 2.0433
   Wall-clock time: 49.0 sec
   Throughput: 28,764 tokens/sec
   Peak GPU Memory: 6285.8 MB

--- Epoch 2/2 ---
Step 0000 | Train Loss: 2.0816 | Val Loss: 2.0738 | lr: 1.352478844e-03
Step 0085 | Train Loss: 1.6620 | Val Loss: 1.6730 | lr: 1.000000000e-03
   Epoch Val Loss: 1.6730
   Wall-clock time: 45.5 sec
   Throughput: 30,956 tokens/sec
   Peak GPU Memory: 6285.8 MB


In [ ]:
import matplotlib.pyplot as plt

# Extract metrics from your XL model results
metrics = mup_training_results[name]
train_losses = metrics["train_loss_history"]
val_losses = metrics["val_loss_history"]
val_steps = metrics["eval_steps"]

plt.figure(figsize=(10, 6))

# Plot training loss (every step)
plt.plot(
    range(len(train_losses)),
    train_losses,
    label="Training Loss",
    color="blue",
    alpha=0.3,
)

# Plot validation loss (only at eval steps)
plt.plot(
    val_steps, val_losses, label="Validation Loss", color="red", marker="o", linewidth=2
)

plt.title("Training and Validation Loss over Time")
plt.xlabel("Training Steps")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()

MuCustomTransformer(
  (token_embedding): Embedding(1000, 768)
  (positional_embedding): Embedding(256, 768)
  (blocks): Sequential(
    (0): MuTransformerBlock(
      (MHA): MuMultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x MuHead(
            (key): Linear(in_features=768, out_features=128, bias=False)
            (query): Linear(in_features=768, out_features=128, bias=False)
            (value): Linear(in_features=768, out_features=128, bias=False)
          )
        )
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ff): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (1): MuTransformerBlock(
  